# 💻 Test Python 10

## 🧰 Carga de todas las librerías necesarias

In [ ]:
# ------------------------------------------------------------------------------
# LIBRERIAS
# ------------------------------------------------------------------------------
import sys
# warnings
import warnings
warnings.filterwarnings('ignore')


# Prueba rápida de carga
try:
    warnings.filterwarnings('ignore')
    import pandas as pd
    import numpy as np
    import great_expectations as ge

    print(f"Python: {sys.version.split()[0]}")
    print(f"Pandas version: {pd.__version__}")
    print(f"NumPy version: {np.__version__}")
    print(f"Great Expectations version: {ge.__version__}")

    print("-" * 80)
    print("🚀 ¡Entorno preparado!")
    print("-" * 80)

except Exception as e:
    print(f"❌ Error de entorno, favor de instalar las dependencias necesarias: {e}")

warnings.filterwarnings('ignore')

# Prueba de Flujo

In [ ]:
import great_expectations as gx
from great_expectations.checkpoint import Checkpoint
from great_expectations.expectations import (
    ExpectColumnValuesToNotBeNull,
    ExpectColumnValuesToBeUnique,
    ExpectColumnValuesToBeOfType
)

# cols
# order_id,order_date,customer_name,city,state,region,country,category,sub_category,product_name,quantity,_unit_price_,_revenue_,_profit_
# expectations
try:
    # 1. Configurar el Contexto de Datos
    # GX 1.x utiliza un contexto para gestionar la configuración y los Data Docs.
    context = gx.get_context()

    # 2. Conectar a los Datos (Data Source y Asset)
    # Definimos el origen de los datos para el archivo que limpiaste previamente.
    datasource_name = "ventas_datasource"
    datasource = context.data_sources.add_pandas(name=datasource_name)

    # Agregamos el archivo limpio como un Asset de datos
    asset_name = "ventas_cleaned_asset"
    asset = datasource.add_csv_asset(
        name=asset_name, 
        filepath_or_buffer="product_sales_cleaned.csv"
    )

    batch_request = asset.build_batch_request()

    #############
    #batch_definition = asset.add_batch_definition(name="default")
    #batch = batch_definition.get_batch()
    #print(batch.columns())
    #############

    # 3. Crear el Suite de Expectativas
    suite_name = "ventas_calidad_suite"
    suite = context.suites.add(gx.ExpectationSuite(name=suite_name))

    # --- DISEÑO DE EXPECTATIVAS (Requerimiento: Mínimo 3) ---

    # Expectativa 1: Unicidad en campos clave (order_id)
    # Garantiza la integridad transaccional eliminando el riesgo de duplicidad.
    suite.add_expectation(ExpectColumnValuesToBeUnique(column="order_id"))

    # Expectativa 2: Presencia y completitud de atributos críticos
    # Validamos que los campos financieros y de cliente no tengan nulos.
    critical_columns = ['order_id', 'customer_name', '_revenue_', '_profit_']
    for col in critical_columns:
        suite.add_expectation(ExpectColumnValuesToNotBeNull(column=col))

    # Expectativa 3: Formatos válidos (Fechas)
    # Verificamos que 'order_date' sea de tipo datetime tras la transformación.
    suite.add_expectation(ExpectColumnValuesToBeOfType(
        column="order_date", 
        type_="datetime64[ns]"
    ))


    # 4. Implementar Punto de Control (Checkpoint)
    # El Checkpoint vincula el asset de datos con el suite de validación.


    ###########
    my_checkpoint_name = "my_databricks_checkpoint"

    checkpoint = Checkpoint(
        name=my_checkpoint_name
    )





except Exception as e:
    print(f"❌ Persiste un error: {e}")

In [ ]:
import pandas as pd
import great_expectations as gx
from great_expectations.expectations import (
    ExpectColumnValuesToNotBeNull,
    ExpectColumnValuesToBeUnique,
    ExpectColumnValuesToBeOfType
)

# 1. Configurar el Contexto
context = gx.get_context()

# 2. Definir Asset de Datos (Usando el archivo limpio)
datasource_name = "ventas_datasource"
datasource = context.data_sources.add_pandas(name=datasource_name)

asset_name = "ventas_cleaned_asset"
asset = datasource.add_csv_asset(
    name=asset_name, 
    filepath_or_buffer="product_sales_cleaned.csv"
)

batch_definition = asset.add_batch_definition(name="default")
batch = batch_definition.get_batch()
print(batch.columns())
print(batch.batch_definition)

# 3. Crear y Guardar el Suite de Expectativas
suite_name = "ventas_calidad_suite"
suite = context.suites.add(gx.ExpectationSuite(name=suite_name))

# Agregar las expectativas al suite
suite.add_expectation(ExpectColumnValuesToBeUnique(column="order_id"))
suite.add_expectation(ExpectColumnValuesToNotBeNull(column="revenue"))
suite.add_expectation(ExpectColumnValuesToBeOfType(
    column="order_date", 
    type_="datetime64[ns]"
))

# 4. PASO CRÍTICO: Crear y añadir la Definición de Validación al Contexto
# Esto registra la unión entre el Asset y el Suite antes de crear el Checkpoint
validation_def_name = "validacion_limpieza_ventas"
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(
        name=validation_def_name,
        data=batch.batch_definition,  # Referencia al Asset registrado
        suite=suite
    )
)

# 5. Configurar el Checkpoint usando la definición ya registrada
checkpoint_name = "ventas_checkpoint"
checkpoint = context.checkpoints.add(
    gx.Checkpoint(
        name=checkpoint_name,
        validation_definitions=[validation_definition] # Pasamos el objeto ya registrado
    )
)

# 6. Ejecutar y generar Documentación
print("Ejecutando validaciones...")
result = checkpoint.run()
context.build_data_docs()

print(f"\n✅ Validación completada: {result.success}")
print(f"📂 Reporte generado exitosamente.")

In [2]:
# -*- coding: utf-8 -*-
"""
Entorno:
- Python: 3.10.19
- pandas: 2.3.3
- numpy: 2.2.6
- great-expectations: 1.14.0

Flujo:
1) Contexto y Data Source (pandas) con CSV asset y parseo de fechas.
2) BatchRequest + Validator.
3) Expectativas (único, not null, dtype datetime64[ns]).
4) Guardar suite, crear/actualizar Checkpoint, ejecutar, Data Docs.
"""

import os
import great_expectations as gx
from great_expectations.checkpoint import Checkpoint

from great_expectations.core.batch_definition import BatchDefinition
from great_expectations.core.validation_definition import ValidationDefinition

#from great_expectations.core.expectation_configuration import ExpectationConfiguration


# 1. Configurar el Contexto
context = gx.get_context()

# 2. Definir Asset de Datos (Usando el archivo limpio)
datasource_name = "ventas_datasource"
datasource = context.data_sources.add_pandas(name=datasource_name)

asset_name = "ventas_cleaned_asset"
asset = datasource.add_csv_asset(
    name=asset_name, 
    filepath_or_buffer="product_sales_cleaned.csv"
)

batch_definition = asset.add_batch_definition(name="default")
batch = batch_definition.get_batch()
#print(batch.columns())
print(batch.batch_definition)


# ---------------------------------------------------------------------
# 2) Suite + Validator
# ---------------------------------------------------------------------

# 3. Crear y Guardar el Suite de Expectativas
suite_name = "ventas_calidad_suite"
suite = context.suites.add(gx.ExpectationSuite(name=suite_name))


# Construye un BatchRequest (API fluida v1.x)
batch_request = asset.build_batch_request()
print(batch_request)


# Obtén un Validator (une datos del batch request + suite)
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite=suite,
)

# (Opcional) Echa un vistazo a columnas disponibles
print("Columnas detectadas:", validator.get_expectation_suite)

# ---------------------------------------------------------------------
# 3) Expectativas (usando el Validator)
# ---------------------------------------------------------------------
# Recomendación: valida existencia de columnas antes de añadir expectativas fuertes

# a) order_id único
validator.expect_column_values_to_be_unique("order_id")

# b) revenue no nulo
validator.expect_column_values_to_not_be_null("_revenue_")

# c) order_date tipo datetime64[ns]
#    OJO: esto verificará el dtype de pandas resultante del parseo
validator.expect_column_values_to_be_of_type("order_date", "datetime64[ns]")

# (Opcional) más reglas útiles:
# - validator.expect_table_row_count_to_be_between(min_value=1)
# - validator.expect_column_values_to_be_between("revenue", min_value=0)
# - validator.expect_column_values_to_match_regex("email", r"^[^@]+@[^@]+\.[^@]+$")

# Guarda/actualiza el suite con las expectativas añadidas
context.suites.add_or_update(gx.ExpectationSuite(name=suite_name, expectations=validator.expectation_suite.expectations))
suite.save()




# 2) Registrar la ValidationDefinition usando el BatchDefinition (no BatchRequest)
validation_def_name = "validacion_limpieza_ventas"
validation_definition = context.validation_definitions.add(
    ValidationDefinition(
        name=validation_def_name,
        data=batch_definition,  # <-- aquí va el BatchDefinition
        suite=suite,
    )
)

# 3) Crear/actualizar el Checkpoint (Fluent) con validation_definitions
checkpoint_name = "ventas_checkpoint"
checkpoint = context.checkpoints.add(
    Checkpoint(
        name=checkpoint_name,
        validation_definitions=[validation_definition],
    )
)

# 4) Ejecutar y Data Docs
print("Ejecutando validaciones...")
result = checkpoint.run()

context.build_data_docs()
for site in context.get_docs_sites_urls() or []:
    print(f"📄 {site.get('site_name')}: {site.get('site_url')}")

print(f"\n✅ Validación completada: {result.success}")

{
  "datasource_name": "ventas_datasource",
  "data_connector_name": "fluent",
  "data_asset_name": "ventas_cleaned_asset",
  "batch_identifiers": {}
}
datasource_name='ventas_datasource' data_asset_name='ventas_cleaned_asset' options={} partitioner=None
Columnas detectadas: <bound method Validator.get_expectation_suite of <great_expectations.validator.validator.Validator object at 0x7293dec27a00>>


/home/idavid/miniconda3/envs/janitor-p10/lib/python3.10/site-packages/great_expectations/expectations/expectation.py:1659: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

/home/idavid/miniconda3/envs/janitor-p10/lib/python3.10/site-packages/great_expectations/expectations/expectation.py:1659: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

/home/idavid/miniconda3/envs/janitor-p10/lib/python3.10/site-packages/great_expectations/expectations/expectation.py:1659: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Ejecutando validaciones...


Calculating Metrics: 0it [00:00, ?it/s]

📄 local_site: file:///tmp/tmpr2ios0o3/index.html

✅ Validación completada: True
